# Scotland population statistics

General statistics for people in Scotland from the enhanced FRS 2023-24 dataset.

In [104]:
import microdf as mdf
import numpy as np
import pandas as pd
from policyengine_uk import Microsimulation

YEAR = 2026
DATASET = "hf://policyengine/policyengine-uk-data/enhanced_frs_2023_24.h5"
sim = Microsimulation(dataset=DATASET)

# Person-level Scotland filter
region_person = sim.calculate("region", YEAR, map_to="person")
is_scot = np.array(region_person) == "SCOTLAND"
person_w = np.array(region_person.weights)[is_scot]

# Household-level Scotland filter
region_hh = sim.calculate("region", YEAR)
is_scot_hh = np.array(region_hh) == "SCOTLAND"
hh_w = np.array(region_hh.weights)[is_scot_hh]


def scot_person_var(var_name):
    """Calculate a variable and return Scotland-only values as a numpy array."""
    return np.array(sim.calculate(var_name, YEAR))[is_scot]


def scot_hh_var(var_name):
    """Calculate a household variable and return Scotland-only values."""
    return np.array(sim.calculate(var_name, YEAR))[is_scot_hh]


print(f"Scotland sample: {is_scot.sum():,} people in {is_scot_hh.sum():,} households")
pop = mdf.MicroSeries(np.ones(is_scot.sum()), weights=person_w)
print(f"Weighted population: {pop.sum():,.0f}")

Scotland sample: 10,542 people in 5,428 households
Weighted population: 5,752,059


## Age distribution

In [105]:
bins = [0, 16, 25, 35, 45, 55, 65, 75, 120]
labels = ["0-15", "16-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75+"]
age_band = pd.cut(scot_person_var("age"), bins=bins, labels=labels, right=False)

sp = mdf.MicroDataFrame({"age_band": age_band, "one": 1}, weights=person_w)
age_dist = sp.groupby("age_band", observed=False).sum().one
age_pct = (age_dist / age_dist.sum() * 100).round(1)

pd.DataFrame({"Population": age_dist.apply(lambda x: f"{x:,.0f}"), "%": age_pct})

,Population,%
age_band,,
0-15,"947,996",16.5
16-24,"676,768",11.8
25-34,"758,158",13.2
35-44,"837,616",14.6
45-54,"708,221",12.3
55-64,"737,273",12.8
65-74,"576,161",10.0
75+,"509,867",8.9


## Gender split

In [106]:
sp = mdf.MicroDataFrame({"gender": scot_person_var("gender"), "one": 1}, weights=person_w)
gender_dist = sp.groupby("gender").sum().one
gender_pct = (gender_dist / gender_dist.sum() * 100).round(1)

pd.DataFrame({"Population": gender_dist.apply(lambda x: f"{x:,.0f}"), "%": gender_pct})

,Population,%
gender,,
FEMALE,"2,843,881",49.4
MALE,"2,908,178",50.6


## Employment status

In [107]:
sp = mdf.MicroDataFrame({"employment_status": scot_person_var("employment_status"), "one": 1}, weights=person_w)
emp_dist = sp.groupby("employment_status").sum().one.sort_values(ascending=False)
emp_pct = (emp_dist / emp_dist.sum() * 100).round(1)

pd.DataFrame({"Population": emp_dist.apply(lambda x: f"{x:,.0f}"), "%": emp_pct})

,Population,%
employment_status,,
FT_EMPLOYED,"2,000,028",34.8
CHILD,"1,118,448",19.4
RETIRED,"852,570",14.8
PT_EMPLOYED,"785,717",13.7
LONG_TERM_DISABLED,"383,686",6.7
FT_SELF_EMPLOYED,"278,047",4.8
UNEMPLOYED,"120,693",2.1
PT_SELF_EMPLOYED,"76,298",1.3
STUDENT,"65,983",1.1


## Marital status

In [108]:
sp = mdf.MicroDataFrame({"marital_status": scot_person_var("marital_status"), "one": 1}, weights=person_w)
mar_dist = sp.groupby("marital_status").sum().one.sort_values(ascending=False)
mar_pct = (mar_dist / mar_dist.sum() * 100).round(1)

pd.DataFrame({"Population": mar_dist.apply(lambda x: f"{x:,.0f}"), "%": mar_pct})

,Population,%
marital_status,,
SINGLE,"3,077,882",53.5
MARRIED,"1,886,446",32.8
WIDOWED,"315,724",5.5
DIVORCED,"291,625",5.1
SEPARATED,"180,383",3.1


## Income summary (annual, adults 16+)

In [109]:
income_cols = [
    "employment_income",
    "self_employment_income",
    "private_pension_income",
    "savings_interest_income",
    "dividend_income",
    "property_income",
]

age = scot_person_var("age")
is_adult = age >= 16
adult_w = person_w[is_adult]

income_data = {col: scot_person_var(col)[is_adult] for col in income_cols}
income_data["total_income"] = sum(income_data.values())
adults = mdf.MicroDataFrame(income_data, weights=adult_w)

print("Total income (adults 16+):")
print(f"  Mean: \u00a3{adults.total_income.mean():,.0f}")
print(f"  Median: \u00a3{adults.total_income.median():,.0f}")
print(f"  25th percentile: \u00a3{adults.total_income.quantile(0.25):,.0f}")
print(f"  75th percentile: \u00a3{adults.total_income.quantile(0.75):,.0f}")
print(f"  Gini coefficient: {adults.total_income.gini():.3f}")

Total income (adults 16+):
  Mean: £26,513
  Median: £23,194
  25th percentile: £3,842
  75th percentile: £36,413
  Gini coefficient: 0.509


## Income by source (weighted mean, adults 16+)

In [110]:
income_means = {}
for col in income_cols:
    income_means[col.replace("_", " ").title()] = f"\u00a3{adults[col].mean():,.0f}"

pd.DataFrame.from_dict(income_means, orient="index", columns=["Weighted mean"])

,Weighted mean
Employment Income,"£20,015"
Self Employment Income,"£1,794"
Private Pension Income,"£3,078"
Savings Interest Income,£600
Dividend Income,£806
Property Income,£220


## Household tenure

In [111]:
sh = mdf.MicroDataFrame({"tenure_type": scot_hh_var("tenure_type"), "one": 1}, weights=hh_w)
tenure_dist = sh.groupby("tenure_type").sum().one.sort_values(ascending=False)
tenure_pct = (tenure_dist / tenure_dist.sum() * 100).round(1)

pd.DataFrame({"Households": tenure_dist.apply(lambda x: f"{x:,.0f}"), "%": tenure_pct})

,Households,%
tenure_type,,
OWNED_OUTRIGHT,"805,571",29.2
OWNED_WITH_MORTGAGE,"606,094",22.0
RENT_PRIVATELY,"551,555",20.0
RENT_FROM_COUNCIL,"422,099",15.3
RENT_FROM_HA,"373,195",13.5


## Data sparsity

How rich is the Scotland sample? For each variable, sparsity measures the weighted proportion of records that are zero or null.

In [112]:
person_vars = [
    "age", "gender", "hours_worked", "marital_status", "current_education",
    "employment_status", "employment_income", "private_pension_income",
    "self_employment_income", "tax_free_savings_income", "savings_interest_income",
    "dividend_income", "property_income", "maintenance_income", "miscellaneous_income",
    "private_transfer_income", "lump_sum_income", "student_loan_repayments",
    "child_benefit_reported", "income_support_reported", "housing_benefit_reported",
    "attendance_allowance_reported", "dla_sc_reported", "dla_m_reported",
    "carers_allowance_reported", "pension_credit_reported",
    "child_tax_credit_reported", "working_tax_credit_reported",
    "state_pension_reported", "universal_credit_reported",
    "pip_m_reported", "pip_dl_reported", "esa_contrib_reported",
    "esa_income_reported", "childcare_expenses", "capital_gains",
]

hh_vars = [
    "region", "tenure_type", "accommodation_type", "council_tax",
    "rent", "mortgage_interest_repayment", "mortgage_capital_repayment",
    "property_wealth", "corporate_wealth", "gross_financial_wealth",
    "net_financial_wealth", "main_residence_value", "savings", "num_vehicles",
    "food_and_non_alcoholic_beverages_consumption",
    "alcohol_and_tobacco_consumption", "clothing_and_footwear_consumption",
    "transport_consumption", "recreation_consumption",
    "domestic_energy_consumption", "petrol_spending", "diesel_spending",
]


def sparsity_report(var_names, fetch_fn, weights):
    rows = []
    for var in var_names:
        vals = fetch_fn(var)
        if vals.dtype.kind in ("f", "i", "u"):
            is_default = (np.isnan(vals) | (vals == 0)).astype(int)
        else:
            is_default = pd.isna(vals).astype(int)
        sparsity = mdf.MicroSeries(is_default, weights=weights).mean() * 100
        rows.append({"Variable": var, "Weighted sparsity %": round(sparsity, 1)})
    return (
        pd.DataFrame(rows)
        .sort_values("Weighted sparsity %", ascending=False)
        .reset_index(drop=True)
    )


print(f"Person variables ({is_scot.sum():,} records)")
display(sparsity_report(person_vars, scot_person_var, person_w))

print(f"\nHousehold variables ({is_scot_hh.sum():,} records)")
display(sparsity_report(hh_vars, scot_hh_var, hh_w))

Person variables (10,542 records)


,Variable,Weighted sparsity %
0,lump_sum_income,100.0
1,working_tax_credit_reported,99.8
2,income_support_reported,99.4
3,esa_income_reported,99.2
4,private_transfer_income,99.0
5,child_tax_credit_reported,99.0
6,dla_m_reported,98.8
7,carers_allowance_reported,98.7
8,dla_sc_reported,98.6
9,esa_contrib_reported,98.5



Household variables (5,428 records)


,Variable,Weighted sparsity %
0,diesel_spending,90.6
1,mortgage_capital_repayment,78.3
2,mortgage_interest_repayment,78.2
3,petrol_spending,58.4
4,rent,57.6
5,main_residence_value,46.2
6,clothing_and_footwear_consumption,45.2
7,property_wealth,43.1
8,savings,36.0
9,alcohol_and_tobacco_consumption,33.9


## Variable categories and data quality

Variables grouped by thematic category, with mean weighted sparsity per category. Lower sparsity = richer data for analysis.

In [113]:
PERSON_CATEGORIES = {
    "Demographics": [
        "age", "gender", "marital_status", "current_education",
        "employment_status", "hours_worked",
    ],
    "Employment & self-employment income": [
        "employment_income", "self_employment_income",
    ],
    "Investment & savings income": [
        "savings_interest_income", "dividend_income", "property_income",
        "tax_free_savings_income", "capital_gains",
    ],
    "Pension income": [
        "private_pension_income", "state_pension_reported",
    ],
    "Other income": [
        "maintenance_income", "miscellaneous_income",
        "private_transfer_income", "lump_sum_income",
    ],
    "Disability & incapacity benefits": [
        "attendance_allowance_reported", "dla_sc_reported", "dla_m_reported",
        "pip_m_reported", "pip_dl_reported",
        "esa_contrib_reported", "esa_income_reported",
    ],
    "Family, housing & means-tested benefits": [
        "child_benefit_reported", "housing_benefit_reported",
        "income_support_reported", "child_tax_credit_reported",
        "working_tax_credit_reported", "universal_credit_reported",
        "pension_credit_reported", "carers_allowance_reported",
    ],
    "Personal expenses": [
        "childcare_expenses", "student_loan_repayments",
    ],
}

HH_CATEGORIES = {
    "Housing characteristics": [
        "region", "tenure_type", "accommodation_type",
    ],
    "Housing costs": [
        "council_tax", "rent",
        "mortgage_interest_repayment", "mortgage_capital_repayment",
    ],
    "Wealth & assets": [
        "property_wealth", "corporate_wealth", "gross_financial_wealth",
        "net_financial_wealth", "main_residence_value", "savings",
    ],
    "Transport": [
        "num_vehicles", "petrol_spending", "diesel_spending",
        "transport_consumption",
    ],
    "Household consumption": [
        "food_and_non_alcoholic_beverages_consumption",
        "alcohol_and_tobacco_consumption",
        "clothing_and_footwear_consumption",
        "recreation_consumption",
        "domestic_energy_consumption",
    ],
}


def category_sparsity(categories, fetch_fn, weights):
    rows = []
    for cat, var_list in categories.items():
        sparsities = []
        for var in var_list:
            vals = fetch_fn(var)
            if vals.dtype.kind in ("f", "i", "u"):
                is_default = (np.isnan(vals) | (vals == 0)).astype(int)
            else:
                is_default = pd.isna(vals).astype(int)
            sparsities.append(
                mdf.MicroSeries(is_default, weights=weights).mean() * 100
            )
        rows.append({
            "Category": cat,
            "Variables": len(var_list),
            "Mean sparsity %": round(np.mean(sparsities), 1),
            "Min sparsity %": round(min(sparsities), 1),
            "Max sparsity %": round(max(sparsities), 1),
        })
    return pd.DataFrame(rows).sort_values("Mean sparsity %").reset_index(drop=True)


print("Person-level categories")
person_cat = category_sparsity(PERSON_CATEGORIES, scot_person_var, person_w)
display(person_cat)

print("\nHousehold-level categories")
hh_cat = category_sparsity(HH_CATEGORIES, scot_hh_var, hh_w)
display(hh_cat)

Person-level categories


,Category,Variables,Mean sparsity %,Min sparsity %,Max sparsity %
0,Demographics,6,8.1,0.0,48.0
1,Employment & self-employment income,2,73.4,52.6,94.2
2,Pension income,2,82.2,82.1,82.3
3,Investment & savings income,5,86.6,57.0,97.6
4,"Family, housing & means-tested benefits",8,96.4,91.1,99.8
5,Personal expenses,2,96.4,95.5,97.3
6,Disability & incapacity benefits,7,97.2,94.3,99.2
7,Other income,4,99.0,98.5,100.0



Household-level categories


,Category,Variables,Mean sparsity %,Min sparsity %,Max sparsity %
0,Housing characteristics,3,0.0,0.0,0.0
1,Household consumption,5,15.9,0.0,45.2
2,Wealth & assets,6,24.6,0.0,46.2
3,Transport,4,48.9,17.8,90.6
4,Housing costs,4,55.5,8.0,78.3


In [114]:
GOOD_THRESHOLD = 30  # mean sparsity % below this = good data

print("=" * 70)
print("SUMMARY: Data quality by category for Scotland")
print("=" * 70)

all_cats = pd.concat([
    person_cat.assign(Level="Person"),
    hh_cat.assign(Level="Household"),
]).sort_values("Mean sparsity %").reset_index(drop=True)

good = all_cats[all_cats["Mean sparsity %"] <= GOOD_THRESHOLD]
sparse = all_cats[all_cats["Mean sparsity %"] > 80]
moderate = all_cats[
    (all_cats["Mean sparsity %"] > GOOD_THRESHOLD)
    & (all_cats["Mean sparsity %"] <= 80)
]

print(f"\nGood data (mean sparsity <= {GOOD_THRESHOLD}%):")
print("These categories are well-suited for distributional analysis.\n")
for _, r in good.iterrows():
    print(f"  {r['Level']:10s} | {r['Category']:45s} | {r['Mean sparsity %']:5.1f}%  ({r['Variables']} vars)")

print(f"\nModerate data ({GOOD_THRESHOLD}-80% sparsity):")
print("Usable for aggregate analysis but thin at the tails.\n")
for _, r in moderate.iterrows():
    print(f"  {r['Level']:10s} | {r['Category']:45s} | {r['Mean sparsity %']:5.1f}%  ({r['Variables']} vars)")

print("\nSparse data (>80% sparsity):")
print("Limited to high-level totals; too few non-zero records for breakdowns.\n")
for _, r in sparse.iterrows():
    print(f"  {r['Level']:10s} | {r['Category']:45s} | {r['Mean sparsity %']:5.1f}%  ({r['Variables']} vars)")

print("\n" + "=" * 70)
print("STRONGEST ANALYSIS AREAS for Scotland:")
print("=" * 70)
print("""
1. Demographics & labour market — age, gender, marital status and
   employment status are near-complete. Strong basis for any
   demographic segmentation or labour market analysis.

2. Employment income distribution — employment income (47% non-zero)
   and hours worked (52% non-zero) support earnings analysis,
   in-work poverty, and tax modelling.

3. Household consumption — food, energy, recreation are near-complete.
   Suitable for indirect tax (VAT) incidence and cost-of-living analysis.

4. Housing — tenure, accommodation type, council tax are well-populated.
   Rent and mortgage data cover the relevant subsets. Good for housing
   policy and council tax reform modelling.

5. Wealth — financial wealth variables have near-zero sparsity.
   Property wealth and savings are moderately populated. Supports
   wealth tax and means-testing analysis.
""")

SUMMARY: Data quality by category for Scotland

Good data (mean sparsity <= 30%):
These categories are well-suited for distributional analysis.

  Household  | Housing characteristics                       |   0.0%  (3 vars)
  Person     | Demographics                                  |   8.1%  (6 vars)
  Household  | Household consumption                         |  15.9%  (5 vars)
  Household  | Wealth & assets                               |  24.6%  (6 vars)

Moderate data (30-80% sparsity):
Usable for aggregate analysis but thin at the tails.

  Household  | Transport                                     |  48.9%  (4 vars)
  Household  | Housing costs                                 |  55.5%  (4 vars)
  Person     | Employment & self-employment income           |  73.4%  (2 vars)

Sparse data (>80% sparsity):
Limited to high-level totals; too few non-zero records for breakdowns.

  Person     | Pension income                                |  82.2%  (2 vars)
  Person     | Investment

## Scotland calibration targets

These are the official statistics that the enhanced FRS household weights are calibrated against for Scotland. They come from `policyengine_uk_data/utils/loss.py` and the CSV/Excel files in `policyengine_uk_data/storage/`.

In [115]:
from policyengine_uk_data.storage import STORAGE_FOLDER

tax_benefit = pd.read_csv(STORAGE_FOLDER / "tax_benefit.csv")
year = str(YEAR)
ct_scotland = tax_benefit[tax_benefit.name == "council_tax_scotland"][year].values[0]

SCP_SPEND = {2024: 455.8, 2025: 471.0, 2026: 484.8}
scp_val = SCP_SPEND.get(YEAR, 471.0 * (1.03 ** (YEAR - 2025)))

summary = [
    {"Category": "Demographics", "Targets": "9 age bands + children + babies + 3+ child HHs", "Count": 12,
     "Source": "NRS, Scotland Census 2022"},
    {"Category": "Council tax total", "Targets": f"\u00a3{ct_scotland}bn", "Count": 1,
     "Source": "OBR March 2024"},
    {"Category": "Council tax bands", "Targets": "Bands A-H + total for SCOTLAND", "Count": 9,
     "Source": "Scottish Government"},
    {"Category": "Scottish Child Payment", "Targets": f"\u00a3{scp_val:.0f}m spend", "Count": 1,
     "Source": "Scottish Budget 2026-27"},
    {"Category": "UC babies", "Targets": "14,000 UC HHs with child <1", "Count": 1,
     "Source": "DWP Stat-Xplore"},
]

print(f"Scotland-specific calibration targets: {sum(r['Count'] for r in summary)}")
print("(Plus Scotland's share of UK-wide targets: income tax, benefits, VAT, etc.)\n")
display(pd.DataFrame(summary))

Scotland-specific calibration targets: 24
(Plus Scotland's share of UK-wide targets: income tax, benefits, VAT, etc.)



,Category,Targets,Count,Source
0,Demographics,9 age bands + children + babies + 3+ child HHs,12,"NRS, Scotland Census 2022"
1,Council tax total,£3.2bn,1,OBR March 2024
2,Council tax bands,Bands A-H + total for SCOTLAND,9,Scottish Government
3,Scottish Child Payment,£485m spend,1,Scottish Budget 2026-27
4,UC babies,"14,000 UC HHs with child <1",1,DWP Stat-Xplore


## Calibration dashboard reference

The microcalibrate dashboard shows how well the enhanced FRS weights match official statistics. Production runs use 512 epochs (logged every 10, so the final entry is epoch 510). The calibration logs are saved as **GitHub Actions artifacts** on each push/PR workflow run.

### How to search for a variable on the dashboard

Target names in the dashboard follow the pattern `{area}/{metric}`, for example:
- `Glasgow City/age/20_30` — population aged 20-30 in Glasgow City
- `City of Edinburgh/hmrc/employment_income/amount` — total employment income in Edinburgh
- `UK/obr/council_tax_scotland` — Scotland-wide council tax total
- `UK/voa/council_tax/SCOTLAND/A` — Scotland council tax band A dwellings

To find a specific variable, type part of the area name or metric into the dashboard search box. The tables below list all Scottish areas and all available metrics.

In [116]:
# --- Scottish local authorities (32) ---
la_areas = {
    "S12000005": "Clackmannanshire", "S12000006": "Dumfries and Galloway",
    "S12000008": "East Ayrshire", "S12000010": "East Lothian",
    "S12000011": "East Renfrewshire", "S12000013": "Na h-Eileanan Siar",
    "S12000014": "Falkirk", "S12000017": "Highland",
    "S12000018": "Inverclyde", "S12000019": "Midlothian",
    "S12000020": "Moray", "S12000021": "North Ayrshire",
    "S12000023": "Orkney Islands", "S12000026": "Scottish Borders",
    "S12000027": "Shetland Islands", "S12000028": "South Ayrshire",
    "S12000029": "South Lanarkshire", "S12000030": "Stirling",
    "S12000033": "Aberdeen City", "S12000034": "Aberdeenshire",
    "S12000035": "Argyll and Bute", "S12000036": "City of Edinburgh",
    "S12000038": "Renfrewshire", "S12000039": "West Dunbartonshire",
    "S12000040": "West Lothian", "S12000041": "Angus",
    "S12000042": "Dundee City", "S12000045": "East Dunbartonshire",
    "S12000047": "Fife", "S12000048": "Perth and Kinross",
    "S12000049": "Glasgow City", "S12000050": "North Lanarkshire",
}

print(f"Scottish local authorities: {len(la_areas)}")
display(
    pd.DataFrame(
        sorted(la_areas.items()), columns=["Code", "Name"]
    ).reset_index(drop=True)
)

# --- Metrics available per local authority (21) ---
la_metrics = {
    "Demographics": [
        "age/0_10", "age/10_20", "age/20_30", "age/30_40",
        "age/40_50", "age/50_60", "age/60_70", "age/70_80",
    ],
    "HMRC income": [
        "hmrc/employment_income/amount", "hmrc/employment_income/count",
        "hmrc/self_employment_income/amount", "hmrc/self_employment_income/count",
    ],
    "ONS income": [
        "ons/equiv_net_income_bhc", "ons/equiv_net_income_ahc",
        "ons/equiv_housing_costs",
    ],
    "Housing": [
        "tenure/owned_outright", "tenure/owned_mortgage",
        "tenure/private_rent", "tenure/social_rent",
        "rent/private_rent",
    ],
    "Benefits": [
        "uc_households",
    ],
}

print("\nMetrics per local authority (21 per area):")
print("Search the dashboard with: {area name}/{metric}")
print()
for category, metrics in la_metrics.items():
    print(f"  {category}:")
    for m in metrics:
        print(f"    {m}")
print()

# --- UK-level Scotland-specific targets ---
uk_scotland_targets = [
    ("Demographics", [
        "UK/ons/scotland_age_0_9", "UK/ons/scotland_age_10_19",
        "UK/ons/scotland_age_20_29", "UK/ons/scotland_age_30_39",
        "UK/ons/scotland_age_40_49", "UK/ons/scotland_age_50_59",
        "UK/ons/scotland_age_60_69", "UK/ons/scotland_age_70_79",
        "UK/ons/scotland_age_80_89",
        "UK/ons/scotland_children_under_16",
        "UK/ons/scotland_households_3plus_children",
    ]),
    ("Council tax", [
        "UK/obr/council_tax_scotland",
        "UK/voa/council_tax/SCOTLAND/A", "UK/voa/council_tax/SCOTLAND/B",
        "UK/voa/council_tax/SCOTLAND/C", "UK/voa/council_tax/SCOTLAND/D",
        "UK/voa/council_tax/SCOTLAND/E", "UK/voa/council_tax/SCOTLAND/F",
        "UK/voa/council_tax/SCOTLAND/G", "UK/voa/council_tax/SCOTLAND/H",
        "UK/voa/council_tax/SCOTLAND/total",
    ]),
    ("Benefits", [
        "UK/dwp/scotland_uc_households_child_under_1",
    ]),
    ("Scottish Child Payment", [
        "UK/sss/scottish_child_payment",
    ]),
]

print("UK-level Scotland targets (search the dashboard with the full name):")
for category, targets in uk_scotland_targets:
    print(f"\n  {category}:")
    for t in targets:
        print(f"    {t}")

Scottish local authorities: 32


,Code,Name
0,S12000005,Clackmannanshire
1,S12000006,Dumfries and Galloway
2,S12000008,East Ayrshire
3,S12000010,East Lothian
4,S12000011,East Renfrewshire
5,S12000013,Na h-Eileanan Siar
6,S12000014,Falkirk
7,S12000017,Highland
8,S12000018,Inverclyde
9,S12000019,Midlothian



Metrics per local authority (21 per area):
Search the dashboard with: {area name}/{metric}

  Demographics:
    age/0_10
    age/10_20
    age/20_30
    age/30_40
    age/40_50
    age/50_60
    age/60_70
    age/70_80
  HMRC income:
    hmrc/employment_income/amount
    hmrc/employment_income/count
    hmrc/self_employment_income/amount
    hmrc/self_employment_income/count
  ONS income:
    ons/equiv_net_income_bhc
    ons/equiv_net_income_ahc
    ons/equiv_housing_costs
  Housing:
    tenure/owned_outright
    tenure/owned_mortgage
    tenure/private_rent
    tenure/social_rent
    rent/private_rent
  Benefits:
    uc_households

UK-level Scotland targets (search the dashboard with the full name):

  Demographics:
    UK/ons/scotland_age_0_9
    UK/ons/scotland_age_10_19
    UK/ons/scotland_age_20_29
    UK/ons/scotland_age_30_39
    UK/ons/scotland_age_40_49
    UK/ons/scotland_age_50_59
    UK/ons/scotland_age_60_69
    UK/ons/scotland_age_70_79
    UK/ons/scotland_age_80_89
    

: 